# NCERT PDF Q&A Chatbot

This notebook implements a production-ready, PDF-based Q&A chatbot that empowers users to upload any NCERT textbook (PDF) and receive accurate, contextually relevant answers strictly based on the uploaded document.

**Inference Only**: No model training or fine-tuning is performed.

**Compatibility**: Designed for Google Colab free-tier.

**Usage**:
1. Upload an NCERT PDF using the widget below.
2. Click **Process PDF** to extract text and build the retrieval index.
3. Enter your question and click **Ask** to get an answer with citations.


In [ ]:
# Install required libraries
!pip install --quiet langchain[all] sentence-transformers faiss-cpu ipywidgets pypdf transformers


In [ ]:
# Imports
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
import transformers
from IPython.display import display, Markdown
import ipywidgets as widgets
from tqdm.auto import tqdm


In [ ]:
# Globals
documents = None
vectorstore = None
qa_chain = None


In [ ]:
# Try to load existing vector store
try:
    embeddings = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
    vectorstore = FAISS.load_local('faiss_index', embeddings)
    print('Loaded existing vector store from disk.')
except Exception:
    print('No existing vector store found. Please upload and process PDFs.')


In [ ]:
def process_pdf(pdf_files):
    """
    Process uploaded PDFs:
    - Save bytes to disk
    - Load and split into pages
    - Chunk text
    - Embed and create FAISS vectorstore
    - Initialize RetrievalQA chain
    """
    global documents, vectorstore, qa_chain
    pdf_paths = []
    for i, data in enumerate(pdf_files):
        path = f'uploaded_{i}.pdf'
        with open(path, 'wb') as f:
            f.write(data)
        pdf_paths.append(path)
    docs = []
    for path in tqdm(pdf_paths, desc='Loading PDFs'):
        try:
            loader = PyPDFLoader(path)
            docs.extend(loader.load_and_split())
        except Exception as e:
            raise ValueError(f'Error loading {path}: {e}')
    if not docs:
        raise ValueError('No text found in uploaded PDFs.')
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    split_docs = splitter.split_documents(docs)
    print(f'Split into {len(split_docs)} chunks.')
    embeddings = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
    vectorstore = FAISS.from_documents(split_docs, embedding=embeddings)
    vectorstore.save_local('faiss_index')
    print('Vector store created and saved.')
    pipe = transformers.pipeline(
        'text2text-generation',
        model='google/flan-t5-base',
        max_length=256,
        temperature=0.0,
    )
    llm = HuggingFacePipeline(pipeline=pipe)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm, chain_type='stuff',
        retriever=vectorstore.as_retriever(),
        return_source_documents=True
    )
    print('QA chain initialized.')


In [ ]:
# User Interface
upload = widgets.FileUpload(accept='.pdf', multiple=True)
process_btn = widgets.Button(description='Process PDF')
question = widgets.Text(description='Your question:')
ask_btn = widgets.Button(description='Ask')
output = widgets.Output()

def on_process(b):
    output.clear_output()
    if not upload.value:
        with output:
            print('Please upload one or more PDF files before processing.')
        return
    pdf_bytes_list = [file['content'] for file in upload.value.values()]
    with output:
        print('Processing PDF(s)...')
        try:
            process_pdf(pdf_bytes_list)
            print('PDFs processed successfully. You can now ask questions.')
        except Exception as e:
            print(f'Error: {e}')

def on_ask(b):
    output.clear_output()
    query = question.value
    if not query.strip():
        with output:
            print('Please enter a valid question.')
        return
    if qa_chain is None:
        with output:
            print('Please process a PDF before asking questions.')
        return
    with output:
        print('Generating answer...')
        res = qa_chain(query)
        answer = res['result']
        src_docs = res['source_documents']
        display(Markdown(f"**Answer:**\n\n{answer}"))
        print('\n**Citations:**')
        for doc in src_docs:
            meta = doc.metadata
            page = meta.get('page', 'Unknown')
            snippet = doc.page_content[:200].strip().replace('\n', ' ')
            print(f'- Page {page}: "{snippet}..."')

process_btn.on_click(on_process)
ask_btn.on_click(on_ask)
display(widgets.VBox([upload, process_btn, question, ask_btn, output]))


In [ ]:
# Example Queries
print("Example queries:")
print("1. What is the definition of Force?")
print("2. Explain the laws of motion.")
print("3. Describe the process of photosynthesis.")


## Testing & Validation
After uploading the sample NCERT PDF and processing, try the example queries above to verify the answers and citations.
Ensure that the answers are accurate and that citations point to the correct pages in the uploaded document.